In [2]:
import kagglehub


path = kagglehub.dataset_download("jessicali9530/caltech256")
print("Path to dataset files:", path)



100%|██████████| 2.12G/2.12G [00:21<00:00, 106MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jessicali9530/caltech256/versions/2


In [3]:
import os
dataset_dir = os.path.join(path, "256_ObjectCategories")


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self,in_channels,out_channels,stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride,bias=False),nn.BatchNorm2d(out_channels))

    def forward(self,x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out

In [5]:
class ResNet18(nn.Module):
    def __init__(self,num_classes):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3,64,kernel_size=7,stride=2,padding=3,bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

        self.layer1 = self._make_layer(64,2,stride=1)
        self.layer2 = self._make_layer(128,2,stride=2)
        self.layer3 = self._make_layer(256,2,stride=2)
        self.layer4 = self._make_layer(512,2,stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512,num_classes)

    def _make_layer(self,out_channels,blocks,stride):
        layers = [BasicBlock(self.in_channels,out_channels,stride)]
        self.in_channels = out_channels
        for _ in range(1,blocks):
            layers.append(BasicBlock(self.in_channels,out_channels))

        return nn.Sequential(*layers)

    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x,1)
        x = self.fc(x)
        return x


In [6]:
if __name__ == '__main__':

    data_directory = dataset_dir
    batch_sizee = 32
    epochs = 20
    learning_rate = 0.001

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing pipeline on device accelerator: {device}")

    data_transforms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
    ])

    full_dataset = datasets.ImageFolder(root=data_directory,transform=data_transforms)

    num_classes = len(full_dataset.classes)
    #split the dataset into 80/20 train/test
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    #LLM help was used for assistance with data loading
    train_dataset,test_dataset = random_split(full_dataset,[train_size,test_size])
    train_loader = DataLoader(train_dataset,batch_size=batch_sizee,shuffle=True)
    test_loader = DataLoader(test_dataset,batch_size=batch_sizee,shuffle=False)

    model = ResNet18(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images,labels in train_loader:

            images,labels = images.to(device),labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs,labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _,predicted = torch.max(outputs,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = 100 * correct / total
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {running_loss/len(train_loader):.4f} "
            f"Train Acc: {train_accuracy:.2f}%"
        )

    model.eval()

    correcte = 0
    totale = 0
    with torch.no_grad():
        for images,labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _,predicted = torch.max(outputs,dim=1)
            totale += labels.size(0)
            correcte += (predicted == labels).sum().item()
    test_accuracy = 100 * correcte / totale
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    },"resnet18_caltech256.pth")
    print("Model weights saved as resnet18_caltech256.pth")

Executing pipeline on device accelerator: cuda
Epoch [1/20] Loss: 5.0834 Train Acc: 7.50%
Epoch [2/20] Loss: 4.4872 Train Acc: 12.84%
Epoch [3/20] Loss: 4.0675 Train Acc: 17.85%
Epoch [4/20] Loss: 3.6784 Train Acc: 23.13%
Epoch [5/20] Loss: 3.3076 Train Acc: 28.55%
Epoch [6/20] Loss: 2.9438 Train Acc: 34.75%
Epoch [7/20] Loss: 2.5968 Train Acc: 40.77%
Epoch [8/20] Loss: 2.2381 Train Acc: 47.41%
Epoch [9/20] Loss: 1.8745 Train Acc: 54.53%
Epoch [10/20] Loss: 1.4843 Train Acc: 62.51%
Epoch [11/20] Loss: 1.0519 Train Acc: 72.40%
Epoch [12/20] Loss: 0.6426 Train Acc: 83.32%
Epoch [13/20] Loss: 0.3492 Train Acc: 91.08%
Epoch [14/20] Loss: 0.2174 Train Acc: 94.61%
Epoch [15/20] Loss: 0.2041 Train Acc: 94.47%
Epoch [16/20] Loss: 0.1774 Train Acc: 95.05%
Epoch [17/20] Loss: 0.1218 Train Acc: 96.72%
Epoch [18/20] Loss: 0.1558 Train Acc: 95.65%
Epoch [19/20] Loss: 0.1501 Train Acc: 95.51%
Epoch [20/20] Loss: 0.0926 Train Acc: 97.32%
Test Accuracy: 35.33%
Model weights saved as resnet18_caltech25